# 🚀 Модуль 7. Урок 4. Проблема подглядывания (Peeking Problem) и P-hacking в аналитике

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_07_ab_testing/lesson_04_p_hacking_peeking_problem.ipynb)

## 📌 Бизнес-контекст и методология
Менеджер проекта запускает A/B тест и каждое утро заглядывает в дашборд. На 4-й день он видит «p-value = 0.03!» и радостно кричит: «Победа, останавливаем тест и катим в прод!».
Это называется **проблемой подглядывания (Peeking Problem)**. 
При ежедневной проверке реальный уровень ложных тревог (False Positive Rate) вырастает с номинальных 5% до **30-40%**! Команда катит в прод фичи-пустышки, удивляясь, почему годовая выручка не растет.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    urllib.request.urlretrieve(url, filename)
    return filename

ab_csv = ensure_dataset("ab_test_checkout.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")

🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных
Смоделируем 500 A/A тестов (где эффект строго равен нулю), в которых аналитик подглядывает результаты каждый день в течение 30 дней.

In [2]:
from scipy import stats
import numpy as np

np.random.seed(42)
n_simulations = 500
n_days = 30
daily_users = 100

peeking_false_positives = 0
final_day_false_positives = 0

for _ in range(n_simulations):
    data_a = np.random.normal(10, 2, n_days * daily_users)
    data_b = np.random.normal(10, 2, n_days * daily_users)
    
    peeked_significant = False
    for day in range(1, n_days + 1):
        sample_a = data_a[:day * daily_users]
        sample_b = data_b[:day * daily_users]
        _, p = stats.ttest_ind(sample_a, sample_b)
        if p < 0.05:
            peeked_significant = True
            break
            
    if peeked_significant:
        peeking_false_positives += 1
        
    _, p_final = stats.ttest_ind(data_a, data_b)
    if p_final < 0.05:
        final_day_false_positives += 1

print(f"Честный тест (проверка строго в день 30): {final_day_false_positives / n_simulations:.1%} ложных тревог (около 5%)")
print(f"Подглядывание (остановка при первом p < 0.05): {peeking_false_positives / n_simulations:.1%} ЛОЖНЫХ ПОБЕД!")

Честный тест (проверка строго в день 30): 5.2% ложных тревог (около 5%)
Подглядывание (остановка при первом p < 0.05): 26.2% ЛОЖНЫХ ПОБЕД!


### 🎬 Анимация: Как подглядывание раздувает ошибку I рода
Запустите анимацию ниже: смотрите, как с каждым новым днем наблюдений накопленный процент ошибочно объявленных «побед» непрерывно растет, взлетая выше 30%!

In [3]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(10, 5))

np.random.seed(42)
days_eval = list(range(1, 31))
cum_fpr_by_day = []

for max_d in days_eval:
    fp = 0
    for _ in range(300):
        data_a = np.random.normal(10, 2, max_d * 80)
        data_b = np.random.normal(10, 2, max_d * 80)
        for d in range(1, max_d + 1):
            _, p = stats.ttest_ind(data_a[:d*80], data_b[:d*80])
            if p < 0.05:
                fp += 1
                break
    cum_fpr_by_day.append(fp / 300 * 100)

def update_peeking(frame):
    ax.clear()
    current_days = days_eval[:frame+1]
    current_fpr = cum_fpr_by_day[:frame+1]
    
    ax.plot(current_days, current_fpr, color='#cc4b37', linewidth=2.5, marker='o', label='Фактический FPR при подглядывании')
    ax.axhline(5.0, color='#3adb76', linestyle='--', linewidth=2, label='Допустимый порог α = 5%')
    ax.set_title(f'Эрозия доверия к тесту: День {current_days[-1]}, FPR = {current_fpr[-1]:.1f}%!', fontsize=12, fontweight='bold')
    ax.set_xlabel('Количество дней ежедневных проверок')
    ax.set_ylabel('Вероятность ложной победы (%)')
    ax.set_xlim(1, 30)
    ax.set_ylim(0, 40)
    ax.legend(loc='upper left')
    ax.grid(True, linestyle=':', alpha=0.6)

anim_peeking = animation.FuncAnimation(fig, update_peeking, frames=len(days_eval), repeat=True)
plt.close(fig)

HTML(anim_peeking.to_jshtml())

### 🎯 Практическое задание 1: Поправка Бонферрони при множественных проверках
Если аналитик обязан проверить тест 5 раз в течение месяца, скорректируйте критический уровень $\alpha = 0.05$ с помощью поправки Бонферрони.

In [4]:
### ВАШ КОД ЗДЕСЬ
k_checks = 5
alpha_bonferroni = 0.05 / k_checks

print(f"Скорректированный порог альфа Бонферрони: {alpha_bonferroni:.4f}")
assert alpha_bonferroni == 0.01, "Поправка Бонферрони: 0.05 / 5 = 0.01"
print("✅ Тест успешно пройден!")

Скорректированный порог альфа Бонферрони: 0.0100
✅ Тест успешно пройден!


### 💼 Как бороться с Peeking Problem в современной индустрии?
1. **Строгое правило фиксации выборки:** Заранее рассчитать $N$, спрятать дашборд и не открывать до набора выборки.
2. **Последовательное тестирование (Sequential Testing):** Использовать метод Вальда (mSPRT) или байесовский подход, где границы решений динамически корректируются с учетом количества наблюдений.